# EN Augmentation Training (upward alignment)

**3处修改:**
1. `make_dataframe()` 训练路径 -> `all_train_articles_en677`
2. `S3_PREFIX` -> `multi_label_models/en_augmentation_677`
3. checkpoint文件名 -> `en677_dual_encoder.ckpt`

超参数不变: xlm-roberta-base, epochs=10, batch=8x4=32, lr=1e-5, loss=bce


In [ ]:
# ============================================================
# CONFIGURATION — update these paths before running
# ============================================================
import os

# Root directory: contains data/, results/, technique_list/, etc.
BASE = "your/path/here"  # e.g. "/home/user/propaganda-span-detection"

# Training data directories
TRAIN_ARTICLES_DIR = "your/path/here"  # multilingual training articles
TRAIN_LABELS_FILE  = "your/path/here"  # corresponding label file
DEV_ARTICLES_DIR   = "your/path/here"  # dev articles
DEV_LABELS_FILE    = "your/path/here"  # dev labels

# SemEval-2023 dev data (per-language evaluation)
SEMEVAL_DEV_DIR = "your/path/here"  # contains en/, po/, ru/ subdirs

# S3 credentials (or set as environment variables)
os.environ.setdefault("AWS_ACCESS_KEY_ID",     "your-access-key-id")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "your-secret-access-key")
S3_ENDPOINT = "https://your-s3-endpoint"
S3_BUCKET   = "your-s3-bucket"


In [ ]:
import pandas as pd
from tqdm import tqdm
import os
from os.path import isfile, join
import boto3
import tempfile
from transformers import AutoModel

os.environ["TOKENIZERS_PARALLELISM"] = 'false'
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

import torch
import numpy as np
import torch.nn.functional as F

import pytorch_lightning as pl
from transformers import get_linear_schedule_with_warmup, AutoTokenizer, AutoModelForSequenceClassification
from transformers import AutoConfig
import torch.nn as nn
from torch.utils.data import DataLoader, RandomSampler
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("Imports done")


## Hyperparameter configuration (change 2: S3 prefix)


In [ ]:
model_name = 'xlm-roberta-base'
EPOCHS = 10
BATCH_SIZE = 8
ACCUMULATE_GRAD_BATCHES = 4
MAX_LENGTH = 256
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.001
WARMUP_STEPS = 1000
LOSS_TYPE = 'bce'
MAX_EXPLANATION_LENGTH = 128

ENDPOINT = "https://your-s3-endpoint"
S3_BUCKET = "your-s3-bucket"
# ===== Change 2: S3 prefix =====
S3_PREFIX = "multi_label_models/en_augmentation_677"
AWS_ACCESS_KEY_ID = os.environ.get("AWS_ACCESS_KEY_ID", "your-access-key-id")
AWS_SECRET_ACCESS_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY", "your-secret-access-key")

print(f"S3_PREFIX: {S3_PREFIX}")
print(f"Batch: {BATCH_SIZE}x{ACCUMULATE_GRAD_BATCHES}={BATCH_SIZE*ACCUMULATE_GRAD_BATCHES}")


In [ ]:
def upload_to_s3(local_file_path, bucket_name, s3_key):
    try:
        s3 = boto3.resource('s3', endpoint_url=ENDPOINT,
            aws_access_key_id=AWS_ACCESS_KEY_ID,
            aws_secret_access_key=AWS_SECRET_ACCESS_KEY)
        bucket = s3.Bucket(bucket_name)
        print(f"Uploading to {ENDPOINT}/{bucket_name}/{s3_key}...")
        bucket.upload_file(local_file_path, s3_key)
        s3_uri = f"{ENDPOINT}/{bucket_name}/{s3_key}"
        print(f"Done: {s3_uri}")
        return True, s3_uri
    except Exception as e:
        print(f"Upload failed: {str(e)}")
        return False, None

def download_from_s3(bucket_name, s3_key, local_file_path):
    try:
        s3 = boto3.resource('s3', endpoint_url=ENDPOINT,
            aws_access_key_id=AWS_ACCESS_KEY_ID,
            aws_secret_access_key=AWS_SECRET_ACCESS_KEY)
        os.makedirs(os.path.dirname(local_file_path), exist_ok=True)
        s3.Bucket(bucket_name).download_file(s3_key, local_file_path)
        return True
    except Exception as e:
        print(f"Download failed: {str(e)}")
        return False

print("S3 functions defined")


## S3 Checkpoint (修改3: 文件名)


In [ ]:
class S3ModelCheckpoint(pl.Callback):
    def __init__(self, bucket_name, s3_prefix, model_name, monitor='val_f1_micro', mode='max'):
        super().__init__()
        self.bucket_name = bucket_name
        self.s3_prefix = s3_prefix
        self.model_name = model_name
        self.monitor = monitor
        self.mode = mode
        self.best_model_score = None
        self.best_model_path = None
        self.temp_dir = tempfile.mkdtemp()
        self.compare = lambda x, y: x > y if self.mode == 'max' else x < y

    def on_validation_end(self, trainer, pl_module):
        current_score = trainer.callback_metrics.get(self.monitor)
        if current_score is None:
            return
        if isinstance(current_score, torch.Tensor):
            current_score = current_score.item()
        if self.best_model_score is None or self.compare(current_score, self.best_model_score):
            self.best_model_score = current_score
            # ===== Change 3: checkpoint filename =====
            filename = f"{self.model_name.split('/')[-1]}_{EPOCHS}_{LOSS_TYPE}_en677_dual_encoder.ckpt"
            temp_path = os.path.join(self.temp_dir, filename)
            trainer.save_checkpoint(temp_path)
            s3_key = f"{self.s3_prefix}/{filename}"
            success, s3_uri = upload_to_s3(temp_path, self.bucket_name, s3_key)
            if success:
                self.best_model_path = s3_uri
                print(f"Best model: {s3_uri} (score: {current_score:.4f})")
                os.remove(temp_path)
    
    def on_train_end(self, trainer, pl_module):
        import shutil
        try: shutil.rmtree(self.temp_dir)
        except: pass

print("S3ModelCheckpoint defined (en677_dual_encoder.ckpt)")


In [ ]:
torch.manual_seed(42)
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"Using: {device} - {torch.cuda.get_device_name(0)}")
    torch.cuda.empty_cache()
else:
    device = torch.device('cpu')
    print("Using CPU")


## 数据加载 (修改1: 训练路径)


In [ ]:
def make_dataframe(data_type='train'):
    # ===== Change 1: training data path =====
    if data_type == 'train':
        input_folder = "your/path/here"
        labels_fn = "your/path/here"
    elif data_type == 'dev':
        input_folder = "your/path/here"
        labels_fn = "your/path/here"
    else:
        raise ValueError("data_type must be 'train' or 'dev'")
    
    print(f"Loading {data_type}: {input_folder}")
    
    text = []
    skipped = 0
    for fil in tqdm(filter(lambda x: x.endswith('.txt'), os.listdir(input_folder))):
        iD = fil.split('.')[0]
        try:
            with open(input_folder+fil, 'r', encoding='utf-8') as f:
                content = f.read()
            lines = list(enumerate(content.splitlines(), 1))
            text.extend([(iD,) + line for line in lines])
        except UnicodeDecodeError:
            skipped += 1
    if skipped:
        print(f"Skipped {skipped} files")

    df_text = pd.DataFrame(text, columns=['id','line','text'])
    df_text.line = df_text.line.apply(int)
    df_text = df_text[df_text.text.str.strip().str.len() > 0].copy()
    df_text = df_text.set_index(['id','line'])

    try:
        labels_data = []
        with open(labels_fn, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line: continue
                parts = line.split('\t')
                if len(parts) < 2: parts = line.split()
                if len(parts) >= 2:
                    try:
                        labels_data.append([parts[0], int(parts[1]), parts[2] if len(parts)>=3 else ''])
                    except ValueError:
                        pass
        if labels_data:
            labels = pd.DataFrame(labels_data, columns=['id','line','labels'])
            labels['line'] = labels['line'].astype(int)
            labels = labels.set_index(['id','line'])
            df = df_text.join(labels, how='left')[['text','labels']]
            df['labels'] = df['labels'].fillna('')
        else:
            raise ValueError("Empty labels")
    except Exception as e:
        print(f"Label error: {e}")
        df = df_text.copy()
        df['labels'] = ''
        df = df[['text','labels']]
    
    df['text'] = df['text'].str.replace(r'\s+', ' ', regex=True).str.strip()
    return df.reset_index()

print("make_dataframe() defined -> en677 paths")


In [ ]:
TECHNIQUES_FILE = "your/path/here"

def load_label_classes():
    with open(TECHNIQUES_FILE, "r") as f:
        labels_name = [line.rstrip() for line in f if line.rstrip()]
    labels_name.sort()
    print(f"Loaded {len(labels_name)} techniques from {TECHNIQUES_FILE}")
    return labels_name

def load_explanations_data(explanations_file):
    explanations = {}
    try:
        df = pd.read_csv(explanations_file, sep='\t')
        for _, row in df.iterrows():
            explanations[(row['id'], row['text'])] = row['analysis']
    except Exception as e:
        print(f"Error: {e}")
    print(f"Loaded {len(explanations)} explanations")
    return explanations

labels = load_label_classes()
print(f"Labels: {labels[:3]}... total={len(labels)}")


In [ ]:
def load_multi_label_data_with_explanations(data_type='train', explanations=None):
    df = make_dataframe(data_type=data_type)
    df = df[df['text'].str.len() <= 1000].copy()
    
    all_idxs = df["id"].to_numpy()
    all_lines = df["line"].to_numpy()
    all_data = [str(t) for t in df["text"].to_numpy()]
    
    labels_name = load_label_classes()
    num_labels = len(labels_name)
    
    multi_labels = []
    for label_str in df['labels'].fillna('').values:
        label_vec = np.zeros(num_labels, dtype=np.float32)
        if label_str:
            for label in label_str.split(','):
                if label in labels_name:
                    label_vec[labels_name.index(label)] = 1.0
        multi_labels.append(label_vec)
    
    if explanations:
        explanation_texts = [explanations.get((idx, text), "") for idx, text in zip(all_idxs, all_data)]
    else:
        explanation_texts = [""] * len(all_data)
    explanation_texts = [str(e) if e else "" for e in explanation_texts]
    
    return (np.array(all_idxs, dtype=object), np.array(all_lines, dtype=np.int32),
            all_data, torch.tensor(np.array(multi_labels)), explanation_texts, labels_name)

print("load_multi_label_data_with_explanations() defined")


In [ ]:
class MultiLabelClassificationDataWithExplanations(torch.utils.data.Dataset):
    def __init__(self, tokenizer=None, max_length=MAX_LENGTH, max_explanation_length=MAX_EXPLANATION_LENGTH,
                 data_tuple=None, data_type='train'):
        self.max_length = max_length
        self.max_explanation_length = max_explanation_length
        if data_tuple is None:
            self.idxs, self.lines, self.texts, self.y, self.explanations, self.label_names = load_multi_label_data_with_explanations(data_type)
        else:
            self.idxs, self.lines, self.texts, self.y, self.explanations, self.label_names = data_tuple
        self.tokenized = False
        if tokenizer is not None:
            self.tokenized = True
            bs = 256
            self.texts = [str(t) for t in self.texts]
            self.text_input_ids, self.text_attention_mask = [], []
            for i in range(0, len(self.texts), bs):
                tok = tokenizer(self.texts[i:i+bs], padding="max_length", truncation=True, max_length=self.max_length, return_tensors="pt")
                self.text_input_ids.append(tok["input_ids"])
                self.text_attention_mask.append(tok["attention_mask"])
            self.text_input_ids = torch.cat(self.text_input_ids, dim=0)
            self.text_attention_mask = torch.cat(self.text_attention_mask, dim=0)
            
            self.exp_input_ids, self.exp_attention_mask = [], []
            exps = [e if e and e.strip() else "no explanation" for e in self.explanations]
            for i in range(0, len(exps), bs):
                tok = tokenizer(exps[i:i+bs], padding="max_length", truncation=True, max_length=self.max_explanation_length, return_tensors="pt")
                self.exp_input_ids.append(tok["input_ids"])
                self.exp_attention_mask.append(tok["attention_mask"])
            self.exp_input_ids = torch.cat(self.exp_input_ids, dim=0)
            self.exp_attention_mask = torch.cat(self.exp_attention_mask, dim=0)
            if hasattr(torch.cuda, 'empty_cache'): torch.cuda.empty_cache()

    def __getitem__(self, index):
        return (self.text_input_ids[index], self.text_attention_mask[index],
                self.exp_input_ids[index], self.exp_attention_mask[index],
                torch.squeeze(self.y[index]), self.idxs[index],
                torch.tensor(self.lines[index], dtype=torch.int64))

    def __len__(self):
        return self.text_input_ids.shape[0] if self.tokenized else len(self.texts)

print("Dataset class defined")


In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, pos_weight=None, reduction='mean'):
        super().__init__()
        self.alpha, self.gamma, self.pos_weight, self.reduction = alpha, gamma, pos_weight, reduction
    def forward(self, inputs, targets):
        eps = 1e-7
        targets_s = torch.clamp(targets.float(), min=eps, max=1-eps)
        bce = F.binary_cross_entropy_with_logits(inputs, targets_s, pos_weight=self.pos_weight, reduction='none') if self.pos_weight is not None else F.binary_cross_entropy_with_logits(inputs, targets_s, reduction='none')
        pt = torch.exp(-bce)
        fl = self.alpha * (1-pt)**self.gamma * bce
        return torch.mean(fl) if self.reduction=='mean' else (torch.sum(fl) if self.reduction=='sum' else fl)

print("FocalLoss defined")


In [ ]:
class MultiLabelClassifierWithExplanations(pl.LightningModule):
    def __init__(self, plm, num_labels, class_weights=None, learning_rate=LEARNING_RATE,
                 warmup_steps=WARMUP_STEPS, loss_type='bce', focal_gamma=2.0, focal_alpha=1.0):
        super().__init__()
        self.text_encoder = AutoModel.from_pretrained(plm)
        self.explanation_encoder = AutoModel.from_pretrained(plm)
        if hasattr(self.text_encoder, "gradient_checkpointing_enable"):
            self.text_encoder.gradient_checkpointing_enable()
            self.explanation_encoder.gradient_checkpointing_enable()
        self.hidden_size = self.text_encoder.config.hidden_size
        self.num_labels = num_labels
        self.learning_rate = learning_rate
        self.warmup_steps = warmup_steps
        self.loss_type = loss_type
        
        self.text_to_exp_attention = nn.MultiheadAttention(embed_dim=self.hidden_size, num_heads=8, batch_first=True)
        self.exp_to_text_attention = nn.MultiheadAttention(embed_dim=self.hidden_size, num_heads=8, batch_first=True)
        self.fusion_layer = nn.Sequential(
            nn.Linear(self.hidden_size*4, self.hidden_size*2), nn.LayerNorm(self.hidden_size*2),
            nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(self.hidden_size*2, self.hidden_size), nn.LayerNorm(self.hidden_size),
            nn.ReLU(), nn.Dropout(0.1))
        self.classifier = nn.Linear(self.hidden_size, num_labels)
        
        if loss_type == 'focal':
            self.criterion = FocalLoss(alpha=focal_alpha, gamma=focal_gamma, pos_weight=class_weights)
        else:
            self.criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights)
        self.train_preds, self.train_labels = [], []
        self.val_preds, self.val_labels = [], []
    
    def forward(self, text_ids, text_mask, exp_ids=None, exp_mask=None):
        text_out = self.text_encoder(input_ids=text_ids, attention_mask=text_mask, return_dict=True)
        text_h = text_out.last_hidden_state
        text_cls = text_h[:, 0, :]
        if exp_ids is None or exp_mask is None:
            return self.classifier(text_cls)
        exp_out = self.explanation_encoder(input_ids=exp_ids, attention_mask=exp_mask, return_dict=True)
        exp_h = exp_out.last_hidden_state
        exp_cls = exp_h[:, 0, :]
        t2e, _ = self.text_to_exp_attention(query=text_h, key=exp_h, value=exp_h, key_padding_mask=~exp_mask.bool())
        e2t, _ = self.exp_to_text_attention(query=exp_h, key=text_h, value=text_h, key_padding_mask=~text_mask.bool())
        combined = torch.cat([text_cls, exp_cls, t2e[:, 0, :], e2t[:, 0, :]], dim=1)
        return self.classifier(self.fusion_layer(combined))

    def _compute_loss(self, preds, labels):
        if labels.shape != preds.shape:
            if len(labels.shape) < len(preds.shape): labels = labels.view(*preds.shape)
            else: preds = preds.view(*labels.shape)
        labels_s = torch.clamp(labels.float(), min=1e-7, max=1-1e-7)
        return self.criterion(preds, labels_s)

    def training_step(self, batch, batch_idx):
        text_ids, text_mask, exp_ids, exp_mask, labels, _, _ = batch
        preds = self(text_ids=text_ids, text_mask=text_mask, exp_ids=exp_ids, exp_mask=exp_mask)
        loss = self._compute_loss(preds, labels)
        if torch.isnan(loss):
            return {"loss": torch.tensor(1e-5, requires_grad=True, device=loss.device)}
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True)
        with torch.no_grad():
            pp = torch.sigmoid(preds)
            if not torch.isnan(pp).any():
                self.train_preds.extend(pp.detach().cpu().numpy())
                self.train_labels.extend(labels.detach().cpu().numpy())
        if batch_idx % 50 == 0 and hasattr(torch.cuda, 'empty_cache'): torch.cuda.empty_cache()
        return {"loss": loss}

    def validation_step(self, batch, batch_idx):
        text_ids, text_mask, exp_ids, exp_mask, labels, _, _ = batch
        preds = self(text_ids=text_ids, text_mask=text_mask, exp_ids=exp_ids, exp_mask=exp_mask)
        loss = self._compute_loss(preds, labels)
        if torch.isnan(loss):
            return {"val_loss": torch.tensor(0.0, device=self.device)}
        self.log('val_loss', loss, prog_bar=True, on_epoch=True)
        with torch.no_grad():
            pp = torch.sigmoid(preds)
            pl_ = (pp > 0.5).float()
            self.log('val_exact_match', (pl_ == labels).all(dim=1).float().mean(), prog_bar=True)
            if not torch.isnan(pp).any():
                self.val_preds.extend(pp.detach().cpu().numpy())
                self.val_labels.extend(labels.detach().cpu().numpy())
        return {"val_loss": loss}

    def _log_metrics(self, preds_list, labels_list, prefix):
        from sklearn.metrics import f1_score, precision_score, recall_score
        pred_l = (np.array(preds_list) > 0.5).astype(int)
        true_l = np.array(labels_list)
        for avg in ['micro', 'macro']:
            is_prog = (prefix == 'val' and avg == 'micro')
            self.log(f'{prefix}_f1_{avg}', f1_score(true_l, pred_l, average=avg, zero_division=0), prog_bar=is_prog)
            self.log(f'{prefix}_precision_{avg}', precision_score(true_l, pred_l, average=avg, zero_division=0), prog_bar=is_prog)
            self.log(f'{prefix}_recall_{avg}', recall_score(true_l, pred_l, average=avg, zero_division=0), prog_bar=is_prog)
        if prefix == 'val' and preds_list:
            for i in range(self.num_labels):
                lf1 = f1_score(true_l[:, i], pred_l[:, i], zero_division=0)
                if lf1 > 0: self.log(f'val_f1_label_{i}', lf1)

    def on_train_epoch_end(self):
        self._log_metrics(self.train_preds, self.train_labels, 'train')
        self.train_preds, self.train_labels = [], []

    def on_validation_epoch_end(self):
        self._log_metrics(self.val_preds, self.val_labels, 'val')
        self.val_preds, self.val_labels = [], []

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.learning_rate, weight_decay=WEIGHT_DECAY)
        sch = get_linear_schedule_with_warmup(opt, num_warmup_steps=self.warmup_steps, num_training_steps=self.trainer.estimated_stepping_batches)
        return {"optimizer": opt, "lr_scheduler": {"scheduler": sch, "interval": "step", "frequency": 1, "monitor": "val_f1_micro"}}

print("Model class defined")


In [ ]:
def compute_multi_label_class_weights(labels, neg_pos_ratio=3.0, max_weight=30.0):
    pos_counts = labels.sum(dim=0)
    total = labels.shape[0]
    neg_counts = total - pos_counts
    weights = []
    for i in range(labels.shape[1]):
        if pos_counts[i] > 0:
            weights.append(min((neg_counts[i]/pos_counts[i])*neg_pos_ratio, max_weight))
        else:
            weights.append(1.0)
    return torch.tensor(weights, dtype=torch.float32, device=labels.device)

print("compute_multi_label_class_weights() defined")


## 数据预检查


In [ ]:
print("=" * 60)
print("Data Sanity Check")
print("=" * 60)
train_folder = "your/path/here"
train_labels = "your/path/here"

if os.path.exists(train_folder):
    all_f = os.listdir(train_folder)
    en_f = [f for f in all_f if f.startswith('en_')]
    tr_f = [f for f in all_f if f.startswith(('fr_','ge_','it_'))]
    print(f"EN: {len(en_f)}, Translated (target 228): {len(tr_f)}, Total: {len(en_f)+len(tr_f)}")

if os.path.exists(train_labels):
    with open(train_labels) as f:
        lines = [l.strip() for l in f if l.strip()]
    trans_ids = set(l.split('\t')[0] for l in lines if l.split('\t')[0].startswith(('fr_','ge_','it_')))
    print(f"Label lines: {len(lines)}, Trans IDs in labels: {len(trans_ids)}")

print("Check done!")

# Expected article counts (from 实验结果汇总):
#   EN original (en_*): 446 articles
#   EN translated (fr_/ge_/it_): 228 articles (QC-filtered from 231)
#   Total EN: 674 articles  |  Translation ratio: 34%
# Expected result: EN(674, 34% translation) Macro F1 = 36.96%  Micro F1 = 47.68%
# (drops from 41.76%: moderate translation share insufficient to regularise)


## 开始训练


In [ ]:
def train_en677_model(explanations_file=None, loss_type=LOSS_TYPE):
    print("=" * 60)
    print("EN677 Augmentation Training")
    print("=" * 60)
    
    labels_name = load_label_classes()
    num_labels = len(labels_name)
    print(f"Labels: {num_labels}, Loss: {loss_type}")
    
    explanations = None
    if explanations_file and os.path.exists(explanations_file):
        explanations = load_explanations_data(explanations_file)
    
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    temp_log_dir = tempfile.mkdtemp(prefix="lightning_logs_")
    
    print("Loading train data...")
    idxs, lines, X, y, exp_train, _ = load_multi_label_data_with_explanations('train', explanations)
    print(f"Train samples: {len(X)}")
    
    print("Loading dev data...")
    d_idxs, d_lines, d_X, d_y, exp_dev, _ = load_multi_label_data_with_explanations('dev', explanations)
    print(f"Dev samples: {len(d_X)}")
    
    class_weights = compute_multi_label_class_weights(y).to(device)
    
    ds_train = MultiLabelClassificationDataWithExplanations(
        tokenizer=tokenizer, data_tuple=(idxs, lines, X, y, exp_train, labels_name))
    ds_val = MultiLabelClassificationDataWithExplanations(
        tokenizer=tokenizer, data_tuple=(d_idxs, d_lines, d_X, d_y, exp_dev, labels_name))
    
    train_loader = DataLoader(ds_train, batch_size=BATCH_SIZE, sampler=RandomSampler(ds_train), num_workers=2, pin_memory=True)
    val_loader = DataLoader(ds_val, batch_size=BATCH_SIZE*2, num_workers=2, pin_memory=True)
    
    model = MultiLabelClassifierWithExplanations(
        plm=model_name, num_labels=num_labels, class_weights=class_weights,
        learning_rate=LEARNING_RATE, warmup_steps=WARMUP_STEPS, loss_type=loss_type)
    
    s3_cb = S3ModelCheckpoint(bucket_name=S3_BUCKET, s3_prefix=f"{S3_PREFIX}/explanation_enhanced",
                              model_name=model_name, monitor='val_f1_micro', mode='max')
    es_cb = EarlyStopping(monitor='val_f1_micro', patience=3, verbose=True, mode='max')
    
    try:
        import tensorboard
        logger = TensorBoardLogger(save_dir=temp_log_dir, name='en677')
    except ImportError:
        from pytorch_lightning.loggers import CSVLogger
        logger = CSVLogger(save_dir=temp_log_dir, name='en677')
    
    trainer = pl.Trainer(
        max_epochs=EPOCHS, callbacks=[s3_cb, es_cb], precision="32",
        accumulate_grad_batches=ACCUMULATE_GRAD_BATCHES, gradient_clip_val=0.5,
        accelerator="auto", devices=1, enable_progress_bar=True,
        enable_model_summary=True, log_every_n_steps=10, logger=logger)
    
    try:
        model = model.to(device)
        trainer.fit(model, train_loader, val_loader)
        if s3_cb.best_model_path:
            print(f"\nBest model: {s3_cb.best_model_path}")
            print(f"Best val_f1_micro: {s3_cb.best_model_score:.4f}")
    except Exception as e:
        print(f"Training failed: {e}")
        import traceback; traceback.print_exc()
    finally:
        if hasattr(torch.cuda, 'empty_cache'): torch.cuda.empty_cache()
        del model, train_loader, val_loader, ds_train, ds_val
        import gc; gc.collect()
    
    try:
        import shutil; shutil.rmtree(temp_log_dir)
    except: pass
    print("\nDone!")

print("train_en677_model() defined")


In [ ]:
explanations_file = "your/path/here"
print(f"Explanations: {explanations_file}")
print(f"Exists: {os.path.exists(explanations_file)}")

train_en677_model(explanations_file=explanations_file, loss_type=LOSS_TYPE)
